Microsoft Agent Framework Demo using Mistral Large 3 LLM

In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated, Any
from random import randint
from pydantic import Field
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework import Agent, tool, ContextProvider, AgentSession, SessionContext, Executor, WorkflowBuilder, AgentResponseUpdate, WorkflowContext, executor, handler

from typing_extensions import Never

from azure.identity import AzureCliCredential

In [2]:
load_dotenv()

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_AI_DEPLOYMENT_NAME = os.getenv("AZURE_AI_DEPLOYMENT_NAME")

In [3]:
client = OpenAIChatCompletionClient(
    azure_endpoint=AZURE_AI_PROJECT_ENDPOINT,
    model=AZURE_AI_DEPLOYMENT_NAME,
    credential=AzureCliCredential(),
)

agent = Agent(
    client=client,
    name="MistralAgent",
    instructions="You are a friendly assistant. Keep your answers brief.",
)

In [4]:
# Non-streaming: get the complete response at once
result = await agent.run("What is the capital of France?")
print(f"Agent: {result}")

Agent: The capital of France is **Paris**.


In [5]:
# Streaming: receive tokens as they are generated
print("Agent (streaming): ", end="", flush=True)
async for chunk in agent.run("Tell me a one-sentence fun fact.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

Agent (streaming): Honey never spoils—edible honey was found in ancient Egyptian tombs!


Add Tools: Tools let your agent call custom functions — like fetching weather data, querying a database, or calling an API.

Define a tool with the @tool decorator:

In [6]:
# NOTE: approval_mode="never_require" is for sample brevity.
# Use "always_require" in production for user confirmation before tool execution.
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."


In [7]:
agent = Agent(
    client=client,
    name="WeatherAgent",
    instructions="You are a helpful weather agent. Use the get_weather tool to answer questions.",
    tools=[get_weather],
)

In [8]:
result = await agent.run("What's the weather like in Seattle?")
print(f"Agent: {result}")

Agent: The current weather in **Seattle** is **stormy** with a high of **18°C**. Would you like any additional details?


Multi-Turn Conversations: Use a session to maintain conversation context so the agent remembers what was said earlier.

In [9]:
# Mistral endpoints reject the `name` field on assistant messages.
# Patch the framework to strip it before sending.
from agent_framework_openai._chat_completion_client import RawOpenAIChatCompletionClient

_orig_prepare = RawOpenAIChatCompletionClient._prepare_message_for_openai

def _prepare_without_assistant_name(self, message):
    msgs = _orig_prepare(self, message)
    for msg in msgs:
        if msg.get("role") == "assistant":
            msg.pop("name", None)
    return msgs

RawOpenAIChatCompletionClient._prepare_message_for_openai = _prepare_without_assistant_name

agent = Agent(
    client=client,
    name="ConversationAgent",
    instructions="You are a friendly assistant. Keep your answers brief.",
)

In [10]:
# Create a session to maintain conversation history
session = agent.create_session()

# First turn
result = await agent.run("My name is Alice and I love hiking.", session=session)
print(f"Agent: {result}\n")

# Second turn — the agent should remember the user's name and hobby
result = await agent.run("What do you remember about me?", session=session)
print(f"Agent: {result}")

Agent: Hi Alice! Nice to meet you. What's your favorite hiking trail?

Agent: I remember your name is Alice and you love hiking! 😊


Memory & Persistence: 
Add context to your agent so it can remember user preferences, past interactions, or external knowledge.

Define a context provider that stores user info in session state and injects personalization instructions:

In [11]:
class UserMemoryProvider(ContextProvider):
    """A context provider that remembers user info in session state."""

    DEFAULT_SOURCE_ID = "user_memory"

    def __init__(self):
        super().__init__(self.DEFAULT_SOURCE_ID)

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Inject personalization instructions based on stored user info."""
        user_name = state.get("user_name")
        if user_name:
            context.extend_instructions(
                self.source_id,
                f"The user's name is {user_name}. Always address them by name.",
            )
        else:
            context.extend_instructions(
                self.source_id,
                "You don't know the user's name yet. Ask for it politely.",
            )

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Extract and store user info in session state after each call."""
        for msg in context.input_messages:
            text = msg.text if hasattr(msg, "text") else ""
            if isinstance(text, str) and "my name is" in text.lower():
                state["user_name"] = text.lower().split("my name is")[-1].strip().split()[0].capitalize()

In [12]:
agent = Agent(
    client=client,
    name="MemoryAgent",
    instructions="You are a friendly assistant.",
    context_providers=[UserMemoryProvider()],
)

In [13]:
session = agent.create_session()

# The provider doesn't know the user yet — it will ask for a name
result = await agent.run("Hello! What's the square root of 9?", session=session)
print(f"Agent: {result}\n")

# Now provide the name — the provider stores it in session state
result = await agent.run("My name is Alice", session=session)
print(f"Agent: {result}\n")

# Subsequent calls are personalized — name persists via session state
result = await agent.run("What is 2 + 2?", session=session)
print(f"Agent: {result}\n")

# Inspect session state to see what the provider stored
provider_state = session.state.get("user_memory", {})
print(f"[Session State] Stored user name: {provider_state.get('user_name')}")


Agent: Hello! I'd be happy to help with that. The square root of 9 is 3. By the way, may I know your name? I'd love to address you properly.

Agent: Hello Alice! It's a pleasure to meet you. How can I assist you today? 😊

Agent: Hello Alice! The sum of 2 + 2 is 4. Is there anything else you'd like to know or calculate? 😊

[Session State] Stored user name: Alice


Workflows let you chain multiple steps together — each step processes data and passes it to the next.

Define workflow steps (executors) and connect them with edges:

In [14]:

# Step 1: A class-based executor that converts text to uppercase
class UpperCase(Executor):
    def __init__(self, id: str):
        super().__init__(id=id)

    @handler
    async def to_upper_case(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Convert input to uppercase and forward to the next node."""
        await ctx.send_message(text.upper())


# Step 2: A function-based executor that reverses the string and yields output
@executor(id="reverse_text")
async def reverse_text(text: str, ctx: WorkflowContext[Never, str]) -> None:
    """Reverse the string and yield the final workflow output."""
    await ctx.yield_output(text[::-1])


def create_workflow():
    """Build the workflow: UpperCase → reverse_text."""
    upper = UpperCase(id="upper_case")
    return WorkflowBuilder(start_executor=upper).add_edge(upper, reverse_text).build()

In [15]:
workflow = create_workflow()

events = await workflow.run("hello world")
print(f"Output: {events.get_outputs()}")
print(f"Final state: {events.get_final_state()}")

Output: ['DLROW OLLEH']
Final state: WorkflowRunState.IDLE
